# Ex. No: 02  
## Adding Sunglasses for a Cool New Look Using OpenCV  

**Name :** H.Kavin prasad    **Reg. No :** 212225040178    **Slot No :** 

#### 1. Import the required libraries.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

#### 2. Read the face image ('face.jpg') and display it.

In [ ]:
face = cv2.imread('face.jpg')
face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

plt.imshow(face_rgb)
plt.title('Original Face Image')
plt.axis('off')
plt.show()
print("Face image shape:", face.shape)

#### 3. Read the sunglasses image ('sunglasses.jpg') and display it.

In [ ]:
glasses = cv2.imread('sunglasses.jpg')
glasses_rgb = cv2.cvtColor(glasses, cv2.COLOR_BGR2RGB)

plt.imshow(glasses_rgb)
plt.title('Sunglasses Image')
plt.axis('off')
plt.show()
print("Sunglasses image shape:", glasses.shape)

#### 4. Detect the face using the Haar Cascade classifier.

In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
print("Faces found:", len(faces))

# Use the largest detected face
fx, fy, fw, fh = max(faces, key=lambda f: f[2] * f[3])

face_box = face_rgb.copy()
cv2.rectangle(face_box, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), 2)
plt.imshow(face_box)
plt.title('Detected Face')
plt.axis('off')
plt.show()

#### 5. Remove the white background of the sunglasses (create a mask) and crop to the frame.

In [ ]:
# Everything that is not (near) white belongs to the sunglasses
gray_g = cv2.cvtColor(glasses, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(gray_g, 240, 255, cv2.THRESH_BINARY_INV)

# Remove tiny specks and close small gaps inside the frame
kernel = np.ones((3, 3), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

# Crop to the bounding box of the sunglasses
ys, xs = np.where(mask > 0)
x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
glasses_crop = glasses_rgb[y1:y2 + 1, x1:x2 + 1]
mask_crop = mask[y1:y2 + 1, x1:x2 + 1]

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1); plt.imshow(glasses_crop);          plt.title('Cropped Sunglasses'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(mask_crop, cmap='gray'); plt.title('Mask');              plt.axis('off')
plt.show()

#### 6. Resize the sunglasses to fit the face and find the position of the eyes.

In [ ]:
# Sunglasses width = face width (a little wider so the frame covers the temples)
new_w = int(fw * 1.0)
scale = new_w / glasses_crop.shape[1]
new_h = int(glasses_crop.shape[0] * scale)

glasses_resized = cv2.resize(glasses_crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
mask_resized = cv2.resize(mask_crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
_, mask_resized = cv2.threshold(mask_resized, 127, 255, cv2.THRESH_BINARY)

# Eyes lie roughly 38% down from the top of the detected face box
eye_y = fy + int(fh * 0.38)
x_pos = fx + (fw - new_w) // 2          # centred horizontally on the face
y_pos = eye_y - new_h // 2              # centred vertically on the eye line

print("Sunglasses size:", glasses_resized.shape)
print("Top-left position (x, y):", x_pos, y_pos)

#### 7. Overlay the sunglasses on the face using the mask.

In [ ]:
result = face_rgb.copy()

roi = result[y_pos:y_pos + new_h, x_pos:x_pos + new_w]

mask_inv = cv2.bitwise_not(mask_resized)
face_part = cv2.bitwise_and(roi, roi, mask=mask_inv)                      # keep face where there is no glasses
glass_part = cv2.bitwise_and(glasses_resized, glasses_resized, mask=mask_resized)  # keep glasses only

result[y_pos:y_pos + new_h, x_pos:x_pos + new_w] = cv2.add(face_part, glass_part)

#### 8. Display the original image and the final image with sunglasses.

In [ ]:
plt.figure(figsize=(10, 7))
plt.subplot(1, 2, 1); plt.imshow(face_rgb); plt.title('Original Image');         plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(result);   plt.title('Image with Sunglasses');  plt.axis('off')
plt.show()

#### 9. Save the final image.

In [ ]:
cv2.imwrite('face_with_sunglasses.jpg', cv2.cvtColor(result, cv2.COLOR_RGB2BGR))
print("Image saved as face_with_sunglasses.jpg")

## Output:
- **i)** Face detected using the Haar Cascade classifier.
- **ii)** Sunglasses background removed using thresholding and masking.
- **iii)** Sunglasses resized and overlaid on the face.

## Result:
Thus, sunglasses were successfully added to the face image using OpenCV face detection, masking and bitwise operations in Python.